In [ ]:
# ================================================
# QA Architect - Modelo de Predicción de Defectos
# Unidad 1 y 2 - Gradiente, SVM, RF, XGBoost, LightGBM
# Unidad 3 - Desbalanceo con SMOTE, métricas avanzadas
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score, average_precision_score,
    precision_recall_curve, roc_curve, matthews_corrcoef, log_loss
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Cargar datos
# =====================

df = pd.read_csv('../data/SoftwareDefectDataset.csv')

print("Shape del dataset:", df.shape)
print("\nPrimeras filas:")
print(df.head())

print("\nDistribución de la clase objetivo:")
print(df['DEFECT_LABEL'].value_counts(normalize=True))

In [ ]:
# 2. Exploración rápida
# =====================
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='DEFECT_LABEL')
plt.title('Distribución de Módulos Defectuosos vs Limpios')
plt.show()

In [ ]:
# 3. Preparación de datos
# =====================

# Separar features y target
X = df.drop(['DEFECT_LABEL'], axis=1)
y = df['DEFECT_LABEL']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos preparados. Train:", X_train.shape, "Test:", X_test.shape)
print("\nDistribución original (train):", dict(zip(*np.unique(y_train, return_counts=True))))

In [ ]:
# 3b. SMOTE - Sobremuestreo de clase minoritaria (TASK-06)
# ======================================================
# La clase minoritaria (DEFECT_LABEL=1) tiene ~33% de los datos.
# SMOTE genera muestras sintéticas para igualar las clases.
# Esto evita que el modelo sesgue hacia la clase mayoritaria.

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Distribución ORIGINAL (train):", dict(zip(*np.unique(y_train, return_counts=True))))
print("Distribución con SMOTE (train):", dict(zip(*np.unique(y_train_smote, return_counts=True))))
print(f"Train shape: {X_train_scaled.shape} -> con SMOTE: {X_train_smote.shape}")

# Visualizar el balanceo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
pd.Series(y_train).value_counts().plot(kind='bar', ax=ax1, color=['skyblue', 'salmon'])
ax1.set_title('Original (train)')
ax1.set_xticklabels(['Limpio (0)', 'Defectuoso (1)'])
pd.Series(y_train_smote).value_counts().plot(kind='bar', ax=ax2, color=['skyblue', 'salmon'])
ax2.set_title('Con SMOTE (train)')
ax2.set_xticklabels(['Limpio (0)', 'Defectuoso (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# 4. Entrenamiento de Modelos (TASK-05: class_weight='balanced')
# ================================================================
# Se comparan dos estrategias contra el desbalanceo:
#   1. class_weight='balanced' (ajusta pesos automáticamente)
#   2. SMOTE (sobremuestreo sintético)
#
# Referencia: Unidad 2 - Sección 3: Manejo de desbalanceo

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                              eval_metric='logloss', n_jobs=-1, scale_pos_weight=2),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                                verbose=-1, n_jobs=-1, class_weight='balanced')
}

# --- Estrategia A: class_weight (datos originales) ---
trained_original = {}
print("=== Estrategia A: class_weight='balanced' (datos originales) ===")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_original[name] = model
    print(f"  >> {name} OK")

# --- Estrategia B: SMOTE (datos balanceados) ---
trained_smote = {}
print("\n=== Estrategia B: SMOTE (datos balanceados) ===")
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    trained_smote[name] = model
    print(f"  >> {name} OK")

In [ ]:
# 5. Evaluación Comparativa (TASK-07: PR-AUC y Curva PR)
# =======================================================
# Se agrega Average Precision (AUC-PR) como métrica principal
# para desbalanceo. La curva PR es más informativa que ROC
# cuando hay clases desbalanceadas (Unidad 2 - Sección 3).

def evaluate_models(trained, X_test, y_test, label=""):
    results = []
    for name, model in trained.items():
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
        mcc = matthews_corrcoef(y_test, y_pred)
        logloss = log_loss(y_test, y_proba)

        results.append({
            'Modelo': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'PR-AUC': pr_auc,
            'MCC': mcc,
            'LogLoss': logloss
        })
    return pd.DataFrame(results).sort_values('PR-AUC', ascending=False)

# Evaluar ambas estrategias
results_original = evaluate_models(trained_original, X_test_scaled, y_test, "Original")
results_smote = evaluate_models(trained_smote, X_test_scaled, y_test, "SMOTE")

print("\n=== Estrategia A: class_weight='balanced' ===")
print(results_original.to_string(index=False))

print("\n=== Estrategia B: Con SMOTE ===")
print(results_smote.to_string(index=False))

# Tablacomparativa lado a lado
comparison = results_original.merge(
    results_smote, on='Modelo', suffixes=('_cw', '_smote')
)
print("\n=== Comparación directa (CW vs SMOTE) ===")
print(comparison[['Modelo', 'PR-AUC_cw', 'PR-AUC_smote', 'Recall_cw', 'Recall_smote']].to_string(index=False))

In [ ]:
# 6. Curvas Precision-Recall (TASK-07)
# ====================================
# La curva PR muestra el trade-off entre precision y recall
# a diferentes thresholds. Útil cuando hay desbalance de clases.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (trained, title) in enumerate([
    (trained_original, 'class_weight=\'balanced\' (Original)'),
    (trained_smote, 'Con SMOTE')
]):
    for name, model in trained.items():
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        axes[ax].plot(rec, prec, label=name)
    axes[ax].set_title(f'Curva PR - {title}')
    axes[ax].legend(loc='lower left')

plt.tight_layout()
plt.show()

# Curvas ROC comparativas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (trained, title) in enumerate([
    (trained_original, 'class_weight=\'balanced\' (Original)'),
    (trained_smote, 'Con SMOTE')
]):
    for name, model in trained.items():
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        axes[ax].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, y_proba):.3f})')
    axes[ax].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[ax].set_title(f'Curva ROC - {title}')
    axes[ax].set_xlabel('FPR')
    axes[ax].set_ylabel('TPR')
    axes[ax].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# 8. Cross-Validation Estratificada (TASK-08)
# ============================================
# Validación más rigurosa usando StratifiedKFold.
# Evalúa cada modelo en 5 folds manteniendo la proporción de clases.
# Referencia: Unidad 2 - Sección 3: Validación cruzada

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print("=== Cross-Validation (5-Fold Estratificada) ===\n")

cv_results = []
for name, model in trained_smote.items():
    scores = cross_validate(model, X_train_scaled, y_train, cv=cv,
                            scoring=scoring_metrics, n_jobs=-1)
    row = {'Modelo': name}
    for metric in scoring_metrics:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    cv_results.append(row)
    print(f">> {name}:")
    for metric in scoring_metrics:
        mean = scores[f'test_{metric}'].mean()
        std = scores[f'test_{metric}'].std()
        print(f"   {metric}: {mean:.4f} +/- {std:.4f}")
    print()

cv_df = pd.DataFrame(cv_results)
print("\n=== Resumen CV ===")
print(cv_df.to_string(index=False))

In [ ]:
# 9. GridSearchCV - Búsqueda de Hiperparámetros (TASK-09)
# ======================================================
# Optimización de hiperparámetros para los mejores modelos.
# Referencia: Unidad 2 - Sección 3: Búsqueda de hiperparámetros

from sklearn.model_selection import GridSearchCV

# --- Logistic Regression: regularización ---
print("=== GridSearch: Logistic Regression ===")
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    lr_params, cv=5, scoring='roc_auc', n_jobs=-1
)
lr_grid.fit(X_train_smote, y_train_smote)
print(f"Mejores params: {lr_grid.best_params_}")
print(f"Mejor ROC-AUC (CV): {lr_grid.best_score_:.4f}")
print(f"ROC-AUC (test): {roc_auc_score(y_test, lr_grid.predict_proba(X_test_scaled)[:,1]):.4f}\n")

# --- Random Forest ---
print("=== GridSearch: Random Forest ===")
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    rf_params, cv=5, scoring='roc_auc', n_jobs=-1
)
rf_grid.fit(X_train_smote, y_train_smote)
print(f"Mejores params: {rf_grid.best_params_}")
print(f"Mejor ROC-AUC (CV): {rf_grid.best_score_:.4f}")
print(f"ROC-AUC (test): {roc_auc_score(y_test, rf_grid.predict_proba(X_test_scaled)[:,1]):.4f}\n")

# --- SVM ---
print("=== GridSearch: SVM (RBF) ===")
svm_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 0.01]
}
svm_grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    svm_params, cv=5, scoring='roc_auc', n_jobs=-1
)
svm_grid.fit(X_train_smote, y_train_smote)
print(f"Mejores params: {svm_grid.best_params_}")
print(f"Mejor ROC-AUC (CV): {svm_grid.best_score_:.4f}")
print(f"ROC-AUC (test): {roc_auc_score(y_test, svm_grid.predict_proba(X_test_scaled)[:,1]):.4f}\n")

# --- XGBoost ---
print("=== GridSearch: XGBoost ===")
xgb_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 6]
}
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1, scale_pos_weight=2),
    xgb_params, cv=5, scoring='roc_auc', n_jobs=-1
)
xgb_grid.fit(X_train_smote, y_train_smote)
print(f"Mejores params: {xgb_grid.best_params_}")
print(f"Mejor ROC-AUC (CV): {xgb_grid.best_score_:.4f}")
print(f"ROC-AUC (test): {roc_auc_score(y_test, xgb_grid.predict_proba(X_test_scaled)[:,1]):.4f}")

# Tabla comparativa de modelos optimizados
print("\n=== Tabla Comparativa: Modelos Originales vs Optimizados ===")
optimized = {
    'Logistic Regression': lr_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_,
    'SVM (RBF)': svm_grid.best_estimator_,
    'XGBoost': xgb_grid.best_estimator_
}
for name, model in optimized.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = model.predict(X_test_scaled)
    print(f"{name:25s} | ROC-AUC: {roc_auc_score(y_test, y_proba):.4f} | "
          f"Precision: {precision_score(y_test, y_pred):.4f} | "
          f"Recall: {recall_score(y_test, y_pred):.4f} | "
          f"F1: {f1_score(y_test, y_pred):.4f}")

In [ ]:
# 10. Curvas de Aprendizaje (TASK-10)
# ====================================
# Diagnóstico de bias/varianza: train vs validation score
# a medida que crece el tamaño del conjunto de entrenamiento.
# Referencia: Unidad 2 - Sección 4: Early Stopping y diagnóstico

from sklearn.model_selection import learning_curve
import numpy as np

def plot_learning_curve(model, X, y, title, ax):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, scoring='roc_auc',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)

    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Train')
    ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Tamaño del conjunto de entrenamiento')
    ax.set_ylabel('ROC-AUC')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

# Seleccionar los 4 modelos principales para curvas de aprendizaje
key_models = {
    'Logistic Regression': trained_smote['Logistic Regression'],
    'Random Forest': trained_smote['Random Forest'],
    'SVM (RBF)': trained_smote['SVM (RBF)'],
    'XGBoost': trained_smote['XGBoost']
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(key_models.items()):
    plot_learning_curve(model, X_train_smote, y_train_smote, name, axes[idx])

plt.suptitle('Curvas de Aprendizaje (SMOTE + class_weight)', fontsize=14)
plt.tight_layout()
plt.show()

print("Interpretación:")
print("- Si train score es alto y validation score bajo -> OVERFITTING (alta varianza)")
print("- Si ambos son bajos -> UNDERFITTING (alto bias)")
print("- Si ambos son altos y cercanos -> modelo equilibrado")

In [ ]:
# 11. Optimización de Threshold de Decisión (TASK-11)
# =====================================================
# En lugar del threshold por defecto (0.5), calculamos F1
# para múltiples umbrales y seleccionamos el óptimo.
# Referencia: Unidad 2 - Sección 3: Curva ROC y umbral óptimo
# Youden's J statistic: J = max(TPR - FPR)

from sklearn.metrics import precision_recall_curve

# Usar el mejor modelo (el de mayor PR-AUC con SMOTE)
best_model_name = results_smote.iloc[0]['Modelo']
best_model = trained_smote[best_model_name]
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Calcular precision, recall y F1 para cada threshold
prec, rec, thresholds = precision_recall_curve(y_test, y_proba)

# F1 para cada threshold (alinear tamaños)
# thresholds tiene un elemento menos que prec/rec
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-10)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

# Youden's J statistic
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
youden_j = tpr - fpr
best_j_idx = np.argmax(youden_j)
best_j_threshold = roc_thresholds[best_j_idx]
best_j = youden_j[best_j_idx]

print(f"Modelo: {best_model_name}")
print(f"\n--- Threshold por defecto (0.5) ---")
y_pred_default = (y_proba >= 0.5).astype(int)
print(f"Precision: {precision_score(y_test, y_pred_default):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_default):.4f}")
print(f"F1: {f1_score(y_test, y_pred_default):.4f}")

print(f"\n--- Threshold óptimo por F1 (max F1={best_f1:.4f}) ---")
print(f"Threshold: {best_threshold:.4f}")
y_pred_f1 = (y_proba >= best_threshold).astype(int)
print(f"Precision: {precision_score(y_test, y_pred_f1):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_f1):.4f}")
print(f"F1: {f1_score(y_test, y_pred_f1):.4f}")

print(f"\n--- Youden's J statistic (max J={best_j:.4f}) ---")
print(f"Threshold: {best_j_threshold:.4f}")
y_pred_j = (y_proba >= best_j_threshold).astype(int)
print(f"Precision: {precision_score(y_test, y_pred_j):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_j):.4f}")
print(f"F1: {f1_score(y_test, y_pred_j):.4f}")

# Gráfico: F1 vs Threshold
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(thresholds, f1_scores, 'b-', linewidth=2)
plt.axvline(x=best_threshold, color='r', linestyle='--', label=f'Optimo F1 ({best_threshold:.3f})')
plt.axvline(x=0.5, color='gray', linestyle=':', label='Default (0.5)')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title(f'F1 vs Threshold - {best_model_name}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(roc_thresholds, youden_j, 'g-', linewidth=2)
plt.axvline(x=best_j_threshold, color='r', linestyle='--', label=f"Optimo J ({best_j_threshold:.3f})")
plt.axvline(x=0.5, color='gray', linestyle=':', label='Default (0.5)')
plt.xlabel('Threshold')
plt.ylabel("Youden's J (TPR - FPR)")
plt.title("Youden's J Statistic")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 13. Tabla Comparativa Final - Todos los Modelos y Métricas (TASK-16)
# ==================================================================
# Tabla unificada con Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, MCC, LogLoss
# para todas las variantes: class_weight (original), SMOTE, y GridSearch optimizados.

from sklearn.metrics import log_loss

def full_evaluation(trained, X_test, y_test, label):
    rows = []
    for name, model in trained.items():
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        rows.append({
            'Variante': label,
            'Modelo': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, y_proba),
            'PR-AUC': average_precision_score(y_test, y_proba),
            'MCC': matthews_corrcoef(y_test, y_pred),
            'LogLoss': log_loss(y_test, y_proba)
        })
    return pd.DataFrame(rows)

# Recolectar todas las variantes
tables = []
tables.append(full_evaluation(trained_original, X_test_scaled, y_test, 'CW (Original)'))
tables.append(full_evaluation(trained_smote, X_test_scaled, y_test, 'SMOTE'))

# Modelos optimizados vía GridSearch (si las variables existen)
try:
    optimized_models = {
        'Logistic Regression': lr_grid.best_estimator_,
        'Random Forest': rf_grid.best_estimator_,
        'SVM (RBF)': svm_grid.best_estimator_,
        'XGBoost': xgb_grid.best_estimator_
    }
    tables.append(full_evaluation(optimized_models, X_test_scaled, y_test, 'GridSearch'))
except NameError:
    print("GridSearch models not available, skipping...")

final_df = pd.concat(tables, ignore_index=True)

print("=" * 120)
print("TABLA COMPARATIVA FINAL - TODOS LOS MODELOS")
print("=" * 120)
print(final_df.to_string(index=False))

# Highlight the best model per metric
print("\n--- Mejor modelo por métrica ---")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC', 'MCC', 'LogLoss']:
    best_row = final_df.loc[final_df[metric].idxmax() if metric != 'LogLoss' else final_df[metric].idxmin()]
    print(f"  {metric:10s}: {best_row['Modelo']:25s} ({best_row['Variante']:15s}) = {best_row[metric]:.4f})" if metric != 'LogLoss' else f"  {metric:10s}: {best_row['Modelo']:25s} ({best_row['Variante']:15s}) = {best_row[metric]:.4f} (menor es mejor)")

In [ ]:
# 7. Feature Importance del Mejor Modelo
# ========================================
# Seleccionamos el mejor modelo (el de mayor PR-AUC con SMOTE)

best_row = results_smote.iloc[0]
best_name = best_row['Modelo']
best_model = trained_smote[best_name]

print(f"Mejor modelo: {best_name}")
print(f"PR-AUC: {best_row['PR-AUC']:.4f} | ROC-AUC: {best_row['ROC-AUC']:.4f}")

if hasattr(best_model, 'coef_'):
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': abs(best_model.coef_[0])
    }).sort_values('importance', ascending=False)
elif hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
else:
    importance = None
    print("Este modelo no soporta feature importance directo.")

if importance is not None:
    print("\nTop características más importantes:")
    print(importance.head(10))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=importance.head(10), x='importance', y='feature', palette='viridis')
    plt.title(f'Importancia de Características - {best_name}')
    plt.tight_layout()
    plt.show()

In [ ]:
# 12. Regularización L1 y L2 en Logistic Regression (TASK-12)
# ===========================================================
# Exploramos cómo los hiperparámetros penalty (L1=Lasso, L2=Ridge)
# y C (inverso de la fuerza de regularización) afectan los coeficientes.
# Referencia: Unidad 2 - Sección 4: Regularización

from sklearn.linear_model import LogisticRegression

# Grid de valores de C (menor C = más regularización)
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
penalties = ['l1', 'l2']

results_reg = []

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, penalty in enumerate(penalties):
    coef_path = []
    for C in C_values:
        lr = LogisticRegression(
            penalty=penalty, C=C, solver='liblinear',
            max_iter=1000, random_state=42, class_weight='balanced'
        )
        lr.fit(X_train_smote, y_train_smote)
        coef_path.append(lr.coef_[0])
        
        y_proba = lr.predict_proba(X_test_scaled)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
        results_reg.append({'Penalty': penalty.upper(), 'C': C, 'ROC-AUC': roc})

    coef_path = np.array(coef_path)
    for i, feature in enumerate(X.columns):
        axes[ax_idx].plot(C_values, coef_path[:, i], 'o-', label=feature)
    axes[ax_idx].set_xscale('log')
    axes[ax_idx].set_xlabel('C (inverso de regularización)')
    axes[ax_idx].set_ylabel('Coeficiente')
    axes[ax_idx].set_title(f'Regularización {penalty.upper()} (L{"asso" if penalty=="l1" else "idge"})')
    axes[ax_idx].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[ax_idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[ax_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla de resultados
reg_df = pd.DataFrame(results_reg)
pivot = reg_df.pivot_table(index='Penalty', columns='C', values='ROC-AUC', aggfunc='first')
print("=== ROC-AUC según C y Penalty ===")
print(pivot.to_string())

print("\n--- Interpretación ---")
print("- C grande (poca regularización): coeficientes grandes,可能 sobreajuste")
print("- C pequeño (mucha regularización): coeficientes cercanos a 0,可能 underfitting")
print("- L1 (Lasso): lleva coeficientes a exactamente cero (selección de features)")
print("- L2 (Ridge): reduce coeficientes pero no a cero")

## Regularización y Control de Overfitting (TASK-13)

### ¿Qué es la regularización?
La regularización es una técnica que **penaliza coeficientes grandes** en el modelo, evitando que
se ajuste demasiado al ruido de los datos de entrenamiento (overfitting). Es el equilibrio entre
**sesgo (bias)** y **varianza (variance)**:

- **Modelo sin regularización**: alta varianza → overfitting → funciona bien en train, mal en test
- **Modelo con demasiada regularización**: alto bias → underfitting → funciona mal en ambos

### L1 (Lasso) vs L2 (Ridge)

| Aspecto | L1 (Lasso) | L2 (Ridge) |
|---------|------------|------------|
| Penaliza | Valor absoluto de coeficientes | Cuadrado de coeficientes |
| Efecto | Lleva coeficientes a **cero** | Reduce coeficientes **sin llegar a cero** |
| Selección de features | Sí (elimina features irrelevantes) | No (mantiene todas) |
| Uso típico | Cuando se sospecha que muchas features son irrelevantes | Cuando todas las features aportan información |

### Parámetro C
En scikit-learn, **C** es el inverso de la fuerza de regularización:
- **C pequeño** → más regularización (coeficientes más pequeños)
- **C grande** → menos regularización (coeficientes más grandes)

### Conexión con Early Stopping (Sección 4, Unidad 2)
Tanto la regularización como el early stopping cumplen la misma función:
**limitar la capacidad del modelo para evitar overfitting**. Mientras la regularización
lo hace añadiendo una penalización a la función de costo, el early stopping
detiene el entrenamiento antes de que el modelo comience a memorizar el ruido.

In [ ]:
# 14. SHAP - Explicabilidad de Predicciones (TASK-17)
# ==================================================
# SHAP (SHapley Additive exPlanations) explica como cada feature
# contribuye a la prediccion individual, basado en teoria de juegos.
# Referencia: Unidad 2 - Post: Explicabilidad de modelos

import shap
import numpy as np

# Usar el mejor modelo entrenado con SMOTE
best_model_name = results_smote.iloc[0]['Modelo']
best_model = trained_smote[best_model_name]
print(f"Generando explicaciones SHAP para: {best_model_name}")

# Configurar explainer segun el tipo de modelo
if hasattr(best_model, 'coef_'):
    # Modelo lineal -> LinearExplainer
    explainer = shap.LinearExplainer(best_model, X_train_smote, feature_perturbation='interventional')
elif hasattr(best_model, 'feature_importances_'):
    # Tree-based -> TreeExplainer
    explainer = shap.TreeExplainer(best_model)
else:
    # Otros -> KernelExplainer (mas lento)
    explainer = shap.KernelExplainer(best_model.predict_proba, X_train_smote[:100])

# Calcular SHAP values para el conjunto de test
shap_values = explainer.shap_values(X_test_scaled)

# Para clasificacion binaria: lista [neg, pos] o array 3D (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_pos = shap_values[:, :, 1]
else:
    shap_values_pos = shap_values

print(f"SHAP values shape: {shap_values_pos.shape}")

# --- Explicacion de predicciones individuales ---
print("\n=== Ejemplos de explicaciones individuales ===")
for idx in [0, 1, 2]:
    print(f"\n--- Modulo {idx} (Real: {'Defectuoso' if y_test.iloc[idx] == 1 else 'Limpio'}) ---")
    prob = best_model.predict_proba(X_test_scaled[idx].reshape(1, -1))[0][1]
    print(f"Probabilidad de defecto: {prob:.2%}")
    
    # Mostrar contribucion de cada feature
    feature_contrib = list(zip(X.columns, shap_values_pos[idx]))
    feature_contrib.sort(key=lambda x: abs(x[1]), reverse=True)
    for feat, val in feature_contrib[:5]:
        direction = "▲ (aumenta riesgo)" if val > 0 else "▼ (disminuye riesgo)"
        print(f"  {feat:20s}: {val:+.4f} {direction}")


In [ ]:
# 15. SHAP Summary Plot (TASK-18)
# ==================================
# Muestra la importancia global de las features con dirección del impacto.
# Cada punto es una observación; color rojo = valor alto de la feature.

import shap

# Gráfico SHAP summary (beeswarm)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_pos, X_test_scaled, feature_names=list(X.columns), show=False)
plt.title(f'SHAP Summary - {best_model_name}', fontsize=14)
plt.tight_layout()
plt.show()

# SHAP bar plot (importancia global)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pos, X_test_scaled, feature_names=list(X.columns),
                  plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance (Global) - {best_model_name}', fontsize=14)
plt.tight_layout()
plt.show()

print("Interpretación del gráfico SHAP Summary:")
print("- Eje X: impacto en la predicción (SHAP value)")
print("- Color: valor de la feature (rojo = alto, azul = bajo)")
print("- Features ordenadas por importancia descendente")
print("- Cada punto = una observación del dataset de test")

In [ ]:
# 8. Matrices de Confusión - Antes vs Después de SMOTE
# ======================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_original.items()):
    y_pred = model.predict(X_test_scaled)
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name} (Original)\nRecall: {recall_score(y_test, y_pred):.3f}')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión - class_weight (Original)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_smote.items()):
    y_pred = model.predict(X_test_scaled)
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name} (SMOTE)\nRecall: {recall_score(y_test, y_pred):.3f}')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión - Con SMOTE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 9. Guardar Mejor Modelo, Scaler, Metricas y Test Data
# ==========================================================
import joblib

# Guardamos el mejor modelo de la estrategia SMOTE
joblib.dump(best_model, '../model/defect_prediction_model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
print(f"Mejor modelo guardado: {best_name}")
print(f"Estrategia: SMOTE + class_weight='balanced'")

# Guardar datos de test para la app
joblib.dump((X_test_scaled, y_test, X.columns.tolist()), '../model/test_data.pkl')
print("Datos de test guardados en model/test_data.pkl")

# Guardar metricas de evaluacion
from sklearn.metrics import log_loss
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
metrics_dict = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba),
    'pr_auc': average_precision_score(y_test, y_proba),
    'mcc': matthews_corrcoef(y_test, y_pred),
    'log_loss': log_loss(y_test, y_proba),
    'model_name': best_name
}
joblib.dump(metrics_dict, '../model/metrics.pkl')
print("Metricas guardadas en model/metrics.pkl")

# Guardar feature importance (con fallback a permutation importance)
if hasattr(best_model, "coef_"):
    importance_dict = dict(zip(X.columns, abs(best_model.coef_[0])))
elif hasattr(best_model, "feature_importances_"):
    importance_dict = dict(zip(X.columns, best_model.feature_importances_))
else:
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(best_model, X_test_scaled[:200], y_test[:200], n_repeats=5, random_state=42)
    importance_dict = dict(zip(X.columns, perm.importances_mean))
joblib.dump(importance_dict, '../model/feature_importance.pkl')
print("Feature importance guardada en model/feature_importance.pkl")

# Guardar tabla comparativa de modelos SMOTE para la app
joblib.dump(results_smote, '../model/comparison_results.pkl')
print("Tabla comparativa guardada en model/comparison_results.pkl")

# Guardar modelos individuales para la pagina Comparacion de Modelos
model_file_map = {
    'Logistic Regression': 'logistic_regression',
    'Random Forest': 'random_forest',
    'SVM (RBF)': 'svm_rbf',
    'XGBoost': 'xgboost',
    'LightGBM': 'lightgbm'
}
for display_name, file_key in model_file_map.items():
    if display_name in trained_smote:
        joblib.dump(trained_smote[display_name], f'../model/model_{file_key}.pkl')
print("Modelos individuales guardados en model/model_*.pkl")


## Conclusiones

### 1. Problema de Desbalanceo de Clases (Sección 3, Unidad 2)

El dataset presenta un desbalance significativo: **67% módulos limpios vs 33% defectuosos**.
Sin técnicas de manejo de desbalanceo, el modelo tiende a predecir siempre la clase mayoritaria,
obteniendo un ROC-AUC cercano a 0.5 (rendimiento aleatorio).

### 2. Soluciones Implementadas

- **`class_weight='balanced'`**: Ajusta los pesos de las clases automáticamente, penalizando más
los errores en la clase minoritaria. Se aplicó en LogisticRegression, SVM, RandomForest y LightGBM.
- **`scale_pos_weight`**: Parámetro específico de XGBoost para balanceo.
- **SMOTE (Synthetic Minority Oversampling Technique)**: Genera muestras sintéticas de la clase
minoritaria interpolando entre vecinos cercanos, balanceando completamente el conjunto de entrenamiento.

### 3. Métricas Clave para Desbalanceo

- **PR-AUC (Average Precision)**: Más informativa que ROC-AUC cuando hay desbalance, porque
se enfoca en el rendimiento sobre la clase positiva (minoritaria).
- **Recall**: Mide qué proporción de defectos reales el modelo logra detectar. Crítico en QA.
- **MCC (Matthews Correlation Coefficient)**: Métrica balanceada que considera las cuatro
categorías de la matriz de confusión.

### 4. Resultados

SMOTE combinado con `class_weight='balanced'` permite al modelo aprender patrones de la clase
minoritaria que antes ignoraba. La curva Precision-Recall muestra claramente la mejora en
la detección de módulos defectuosos.

### 5. Aplicación Práctica para QA

Con un modelo que detecta correctamente módulos defectuosos, los equipos de QA pueden:
- Priorizar el testing en los módulos de alto riesgo
- Optimizar recursos limitados de pruebas
- Implementar Risk-Based Testing basado en evidencia cuantitativa

In [ ]:
# 10. Ejecutar el Dashboard
# ==========================
# Para iniciar la aplicación Streamlit:
# > streamlit run app.py
#
# Asegúrate de estar en el directorio 04.qaDefectPrediction/